# Adapt the Input, Not the Weights
### Input-Path Isolation Prevents Catastrophic Forgetting in Trainable Encoders

**Reproduction notebook.** Every number in the paper is regenerated here, and every
design decision is explained where it was made — including the ones that turned out
to be wrong.

---

## The result in one table

| configuration | AVG | forgetting |
|---|---|---|
| vanilla LSTM | 0.4399 ± 0.0282 | 0.6653 |
| + memory bank / GGC composition (PLCM) | 0.4304 ± 0.0109 | 0.6770 |
| + EWC weight regularization (λ=200) | 0.4330 ± 0.0441 | 0.6718 |
| + experience replay (100 ex/task) | 0.7916 ± 0.0051 | 0.2236 |
| **+ per-task input adapters** | **0.9331 ± 0.0099** | **0.0560** |

All rows n=3, Permuted MNIST, 5 tasks × 10 epochs, task-incremental evaluation.

**The claim:** forgetting in these models is dominated by *input-path* interference,
not weight interference. Routing each task's input through its own learned linear map
recovers +49.3pp, while defending the weights (EWC) recovers nothing measurable, and
the memory system this project was originally built around recovers nothing measurable
either.

## How to use this notebook

Two modes:

- **ANALYZE (default, seconds).** Every results cell reads the saved JSON in `runs/`
  and regenerates the tables. Run the whole notebook top to bottom; nothing trains.
- **REPRODUCE (hours).** Cells marked `# REPRODUCE` contain the exact commands that
  produced those JSONs. Runtime is stated per cell. They are `%%script false --no-raise-error`
  by default so a full top-to-bottom run is safe; delete that first line to actually run one.

---
# 1. Environment

Python 3.11 in `.venv` (the system Python is 3.14, which has no PyTorch wheels).

In [ ]:
import json, os, sys, subprocess
from pathlib import Path
import numpy as np
import torch

ROOT = Path.cwd()
assert (ROOT / "src" / "models" / "plcm.py").exists(), f"run this notebook from the repo root, not {ROOT}"
sys.path.insert(0, str(ROOT))

print("python  ", sys.version.split()[0])
print("torch   ", torch.__version__)
print("device  ", "mps" if torch.backends.mps.is_available() else "cpu")
print("repo    ", ROOT)

In [ ]:
# The data is downloaded by torchvision on first use (checksum-verified, ~63 MB).
from src.data.permuted_mnist import PermutedMNISTBenchmark
bench = PermutedMNISTBenchmark(num_tasks=5, batch_size=128, seed=42)
print("tasks:", bench.num_tasks)
print("task 0 permutation:", bench.permutations[0], "(identity by convention)")
print("task 1 permutation:", bench.permutations[1][:8].tolist(), "...")

---
# 2. The LSTM backbone, in detail

The backbone is a **single-layer LSTM, hidden size 256**. Understanding three specific
choices is necessary to follow everything downstream.

## 2.1 The image is a sequence of rows

A 28×28 MNIST image is fed as **28 timesteps of 28 features** — one row per timestep.
So `input_size=28`, not 784.

This is the single most consequential framing decision in the project, because it
determines what a *permutation* does. Permuted MNIST permutes the **flattened** 784-vector.
After reshaping back to 28×28, a pixel that was in row 3 can land in row 20. So a
permutation **scatters information across timesteps** — maximally destructive to a
recurrent accumulator, and the reason this benchmark is hard for an LSTM specifically.

It is also why an early bug mattered: `configs/default.yaml` initially set
`input_size: 784`, which crashes the LSTM (it receives 28-dim timesteps).

In [ ]:
x, y = next(iter(bench.get_task_loaders(0)[0]))
print("batch shape        ", tuple(x.shape), "-> [batch, timesteps, features_per_step]")

lstm = torch.nn.LSTM(input_size=28, hidden_size=256, batch_first=True)
out, (h_n, c_n) = lstm(x)
print("lstm_out           ", tuple(out.shape),  "-> per-timestep outputs")
print("h_n (hidden)       ", tuple(h_n.shape))
print("c_n (cell)         ", tuple(c_n.shape))
print()
print("PLCM classifies from c_n[-1]:", tuple(c_n[-1].shape))

## 2.2 Why the **cell state**, not the hidden state

PLCM's "thought space" is the LSTM **cell state** `c_n[-1]`, not the hidden output.
The cell state is the accumulator carried across timesteps; the hidden state is a
*gated view* of it (`h = o ⊙ tanh(c)`).

**This cost a full validation cycle to learn.** Synthetic tasks with the class signal
in the *final row* are learnable from a hidden-state readout but **not** reliably from
the cell state, so they fail to train PLCM and sit at chance — making them useless for
validating anything downstream. Any synthetic task used to test this architecture must
spread signal across timesteps.

In [ ]:
# Demonstration: signal in the LAST row only vs. spread across ALL rows.
g = torch.Generator().manual_seed(0)

def last_row_task(n):        # learnable from h, hard from c
    yy = torch.randint(0, 10, (n,), generator=g)
    xx = torch.randn(n, 28, 28, generator=g) * 0.3
    xx[torch.arange(n), -1, yy] += 5.0
    return xx, yy

def all_rows_task(n):        # accumulates -> learnable from c
    yy = torch.randint(0, 10, (n,), generator=g)
    xx = torch.randn(n, 28, 28, generator=g) * 0.3
    xx[torch.arange(n), :, yy] += 3.0
    return xx, yy

for name, fn in [("signal in last row ", last_row_task), ("signal in all rows ", all_rows_task)]:
    xx, yy = fn(256)
    with torch.no_grad():
        o, (hh, cc) = lstm(xx)
    # variance of the class-relevant direction is a crude proxy for "is it in there"
    print(f"{name}: ||c_n|| = {cc[-1].norm(dim=1).mean():.2f}   ||h_n|| = {hh[-1].norm(dim=1).mean():.2f}")
print("\n(The real lesson is empirical: last-row tasks train from h but not from c.)")

## 2.3 Recovering the output gate exactly

The readout needs `h' = o ⊙ tanh(c')` where `c'` is a *modified* cell state. But PyTorch's
LSTM does not expose `o`. It can be recovered **exactly** by inversion:

$$h = o \odot \tanh(c) \;\Longrightarrow\; o = h \,/\, \tanh(c)$$

guarding the division where `tanh(c) ≈ 0`. This is exact, not approximate — verified below.

In [ ]:
with torch.no_grad():
    o, (hh, cc) = lstm(x)
c_t, h_t = cc[-1], hh[-1]
tanh_ct  = torch.tanh(c_t)
gate = torch.where(tanh_ct.abs() > 1e-6, h_t / tanh_ct, torch.ones_like(h_t)).clamp(0, 1)
recon = gate * tanh_ct
ok = tanh_ct.abs() > 1e-3          # non-degenerate dimensions
print(f"max |o*tanh(c) - h| on non-degenerate dims: {(recon[ok]-h_t[ok]).abs().max():.2e}")
print(f"recovered gate range: [{gate.min():.3f}, {gate.max():.3f}]")

---
# 3. The architecture, and every design decision that shaped it

The model is `src/models/plcm.py`. It accumulated nine mechanisms over the project.
**Four of them are measured to contribute nothing.** This section documents what each
was for, and what the evidence said — because the negative results are the paper.

## 3.1 The original thesis (and its fate)

The project began from this idea: store LSTM cell states in a persistent external
**memory bank**, retrieve relevant past states, and compose them with the current state
using a **Gated Geometric Composition** (GGC) operator instead of additive blending:

$$\gamma = \sigma(W_\gamma[c_t; \tilde c; c_t \odot \tilde c]),\quad
\mu = \sigma(W_\mu \tilde c),\quad
\iota = \tanh(W_\iota[c_t;\tilde c])$$
$$c_t' = \gamma \odot (c_t \odot \mu) + (1-\gamma)\odot \iota$$

The reasoning for GGC over additive blending was sound: repeated additive blending
converges to the centroid of stored vectors, destroying task-specific information.

**Measured outcome: the memory system contributes nothing.** Four independent tests:

| test | result |
|---|---|
| PLCM (bank + GGC) vs vanilla LSTM | 0.4304 vs 0.4399 — **−0.95pp, inside noise** |
| memory-off ablation on the adapter config | 0.9798 vs 0.9767 — memory slightly *hurt* |
| task-free routing via the bank | 20% routing accuracy (chance = 20%) |
| RBST state transport through the bank | worse than no-op in **12/12** cells |

The GGC math is correct and the additive-collapse argument is right. The mechanism
simply does not help on this benchmark.

In [ ]:
# The GGC operator itself, verified against its two design properties.
from src.models.composition import GatedGeometricComposition, AdditiveComposition
torch.manual_seed(0)
ggc, add = GatedGeometricComposition(64), AdditiveComposition(64)
c_cur, c_mem = torch.randn(8, 64), torch.randn(8, 64)

out_ab, _ = ggc(c_cur, c_mem)
out_ba, _ = ggc(c_mem, c_cur)
print(f"GGC is asymmetric (order matters): {not torch.allclose(out_ab, out_ba)}")

# additive collapse: repeated blending drifts toward the mean
v, bank = torch.randn(1, 64), torch.randn(32, 64)
for _ in range(30):
    v, _ = add(v, bank[torch.randint(0, 32, (1,))])
cos_to_mean = torch.nn.functional.cosine_similarity(v, bank.mean(0, keepdim=True)).item()
print(f"additive after 30 blends, cosine to bank centroid: {cos_to_mean:+.3f}  (collapse)")

## 3.2 Decision log — what changed and why

Chronological. Each row is a decision, the evidence that forced it, and its status.

| # | decision | evidence | status |
|---|---|---|---|
| 1 | GGC instead of additive composition | additive → centroid collapse (above) | correct, but the whole memory path is inert |
| 2 | Fixed-capacity bank with importance eviction | unbounded growth is not a memory system | kept |
| 3 | Bounded importance decay | v1 decay had a `usage_boost` **multiplier > 1** → importance exploded to ∞, freezing the bank | **bug fixed**; decay now ∈ [base, 1) |
| 4 | Task-aware bank (`rebalance`) with protected slots | bank was a single-task cache: `num_tasks_stored` stuck at 1 | fixed mechanically (1→5); **no accuracy benefit** |
| 5 | Trained write controller | it never receives gradients — importance is a *random projection* | documented failure mode, never fixed |
| 6 | Freeze encoder after task 0 | unfrozen: 97% diagonal / 13% retention. frozen: 25–59% diagonal / 94% retention | the stability–plasticity dilemma, measured |
| 7 | Per-task input adapters **784×784** on the flattened image | a 28×28 per-row adapter **cannot** represent a 784-permutation (it can't move pixels between rows) | the mechanism that works |
| 8 | Adapters do **not** need a frozen encoder | unfrozen + adapters = 0.9331; freezing adds only +4.08pp | corrected the project's own attribution |
| 9 | Low-rank adapters (`A = I + UV`) | rank 32 → −25.9pp, rank 64 → −7.5pp; learned deviation has effective rank 56–74 | rejected: capacity-bound |

### Decision 7 in detail — the shape that decided the project

This is the one place where getting the *shape* wrong would have produced a
convincing false negative.

The permutation acts on the **flattened 784-vector**. A per-row 28×28 adapter applied to
each timestep can only remix the 28 values *within* a row, and applies the same matrix to
every row — so it cannot represent a 784-permutation at all. The demonstration below uses
a **frozen** LSTM (position-sensitive) because a linear probe would hide the difference.

In [ ]:
# Frozen LSTM + permuted input: can a linear adapter restore accuracy?
import torch.nn as nn
torch.manual_seed(0); g = torch.Generator().manual_seed(1)
perm = torch.randperm(784, generator=g)

def task(n, permute):
    yy = torch.randint(0, 10, (n,), generator=g)
    xx = torch.randn(n, 784, generator=g) * 0.3
    xx[torch.arange(n), 756 + yy] += 6.0        # signal lives in row 27
    if permute: xx = xx[:, perm]
    return xx.reshape(n, 28, 28), yy

class Enc(nn.Module):
    def __init__(s):
        super().__init__(); s.l = nn.LSTM(28, 64, batch_first=True); s.h = nn.Linear(64, 10)
    def forward(s, z):
        _, (hh, _) = s.l(z); return s.h(hh[-1])

enc = Enc(); opt = torch.optim.Adam(enc.parameters(), 2e-3)
for _ in range(400):                                  # train on the UNPERMUTED task
    xx, yy = task(64, False); opt.zero_grad()
    nn.functional.cross_entropy(enc(xx), yy).backward(); opt.step()
for p in enc.parameters(): p.requires_grad = False    # FREEZE

def acc(adapt, permute=True, n=20):
    c = t = 0
    with torch.no_grad():
        for _ in range(n):
            xx, yy = task(64, permute); b = xx.shape[0]
            za = adapt(xx.reshape(b, 784)).reshape(b, 28, 28) if adapt else xx
            c += (enc(za).argmax(1) == yy).sum().item(); t += yy.numel()
    return c / t

def train_adapter(A, steps=500):
    o = torch.optim.Adam(A.parameters(), 2e-3)
    for _ in range(steps):
        xx, yy = task(64, True); b = xx.shape[0]; o.zero_grad()
        nn.functional.cross_entropy(enc(A(xx.reshape(b, 784)).reshape(b, 28, 28)), yy).backward(); o.step()
    return A

A784 = nn.Linear(784, 784, bias=False); nn.init.eye_(A784.weight); train_adapter(A784)

class RowAdapter(nn.Module):                           # the WRONG shape
    def __init__(s):
        super().__init__(); s.m = nn.Linear(28, 28, bias=False); nn.init.eye_(s.m.weight)
    def forward(s, z):
        b = z.shape[0]; return s.m(z.reshape(b, 28, 28)).reshape(b, 784)

print(f"frozen LSTM, identity layout      : {acc(None, permute=False):.3f}")
print(f"frozen LSTM, permuted, no adapter : {acc(None):.3f}   (broken)")
print(f"  + 784x784 adapter  (CORRECT)    : {acc(A784):.3f}   <- fully recovers")
print(f"  + 28x28 per-row    (WRONG shape): {acc(train_adapter(RowAdapter())):.3f}   <- cannot un-scramble rows")

## 3.3 What the adapter actually does — *not* what it was designed to do

The adapters were motivated by an exact-inversion argument: the task shift is a
permutation $P_k$, a linear operator, so a linear adapter can represent $P_k^{-1}$ exactly.

**That justification is falsified as an explanation**, by two independent experiments:

1. **Rotated MNIST.** Rotation is linear but *lossy* — a +67.5°/−67.5° round trip destroys
   24.9% of the signal, so no linear adapter can invert it. The adapter arm scores
   **0.9366** there vs **0.9331** on Permuted: it does not care whether an exact inverse exists.
2. **Low-rank adapters.** Rank-64 sits *further* from the full-rank solution (rel. error 1.10)
   than rank-32 (0.82) while scoring **13pp higher**. Fidelity and performance run in
   *opposite* directions — there is no single target transform being approximated.

The mechanism is **per-task learned preprocessing plus gradient isolation**: each task's
gradients reach the shared encoder only through that task's own input map, so the encoder
sees near-identically-distributed streams and has little task-specific structure to overwrite.
Rank buys *expressive capacity*, not fidelity to any particular map.

In [ ]:
# The effective rank of the learned deviation A - I (needs the checkpointed full-rank run).
ck = ROOT / "checkpoints/fullrank_ref/mafc_seed42/task4_epoch9.pt"
if ck.exists():
    from src.models.plcm import PLCM
    m = PLCM.load_from_checkpoint(torch.load(ck, weights_only=False, map_location="cpu"))
    print(f"{'task':>5}{'||A-I||':>10}{'rank@90%':>10}{'rank@99%':>10}{'top-32 energy':>15}")
    for k in sorted(m.task_adapters):
        W = m.task_adapters[k].weight.detach()
        D = W - torch.eye(W.shape[0])
        s = torch.linalg.svdvals(D); e = (s**2).cumsum(0) / (s**2).sum()
        print(f"{k:>5}{D.norm():>10.2f}{int((e<0.90).sum())+1:>10}{int((e<0.99).sum())+1:>10}{float(e[31]):>15.3f}")
    print("\n-> the deviation is genuinely high-rank; rank 32 captures only ~75-80%.")
else:
    print("checkpoint not present; see runs/lowrank/MEMO.md for the recorded table")

---
# 4. Reproducing every experiment

Each block states its runtime and writes JSON into `runs/`. To actually execute one,
**delete the `%%script false` line**. Total for everything: roughly 10–12 hours on an
M-series CPU/MPS.

Common flags: `--model mafc --mafc-arm lambda0` gives the plain configuration
(adapters + unfrozen encoder + shared readout, **no** MAFC loss);
`--no-adapters` is the single-variable control; `--seed` and `--log-dir` isolate runs.

In [ ]:
%%script false --no-raise-error
# REPRODUCE — main table: adapters (unfrozen) and the no-adapter control. ~2h
for S in 42 1337 2024; do
  .venv/bin/python scripts/train.py --config configs/mafc_phase1.yaml \
    --model mafc --mafc-arm lambda0 --seed $S --log-dir runs/mafc_phase1/lambda0_seed$S
  .venv/bin/python scripts/train.py --config configs/mafc_phase1.yaml \
    --model mafc --mafc-arm lambda0 --no-adapters --seed $S \
    --log-dir runs/mafc_attrib/noadapt_seed$S
done

In [ ]:
%%script false --no-raise-error
# REPRODUCE — baselines: vanilla LSTM, EWC lambda=200, experience replay. ~2h
for S in 42 1337 2024; do
  .venv/bin/python scripts/train.py --config configs/default.yaml --model lstm      --seed $S --log-dir runs/lstm_seed$S
  .venv/bin/python scripts/train.py --config configs/default.yaml --model plcm_ewc  --ewc-lambda 200 --seed $S --log-dir runs/ewc_seed$S
  .venv/bin/python scripts/train.py --config configs/mafc_phase1.yaml --model mafc --mafc-arm lambda0 \
    --no-adapters --er --er-buffer 100 --seed $S --log-dir runs/er_seed$S
done

In [ ]:
%%script false --no-raise-error
# REPRODUCE — generality: Rotated MNIST and the MLP backbone, both arms. ~4h
for S in 42 1337 2024; do
  .venv/bin/python scripts/train.py --config configs/mafc_phase1.yaml --model mafc --mafc-arm lambda0 \
    --benchmark rotated --seed $S --log-dir runs/rot_adapt_seed$S
  .venv/bin/python scripts/train.py --config configs/mafc_phase1.yaml --model mafc --mafc-arm lambda0 \
    --benchmark rotated --no-adapters --seed $S --log-dir runs/rot_noadapt_seed$S
  .venv/bin/python scripts/train.py --config configs/mafc_phase1.yaml --model mafc --mafc-arm lambda0 \
    --backbone mlp --seed $S --log-dir runs/mlp_adapt_seed$S
  .venv/bin/python scripts/train.py --config configs/mafc_phase1.yaml --model mafc --mafc-arm lambda0 \
    --backbone mlp --no-adapters --seed $S --log-dir runs/mlp_noadapt_seed$S
done

In [ ]:
%%script false --no-raise-error
# REPRODUCE — storage frontier: low-rank adapters + larger replay buffers. ~3h
for R in 32 64; do for S in 42 1337 2024; do
  .venv/bin/python scripts/train.py --config configs/mafc_phase1.yaml --model mafc --mafc-arm lambda0 \
    --adapter-rank $R --seed $S --log-dir runs/lowrank_r${R}_seed$S
done; done
for B in 200 500; do
  .venv/bin/python scripts/train.py --config configs/mafc_phase1.yaml --model mafc --mafc-arm lambda0 \
    --no-adapters --er --er-buffer $B --seed 42 --log-dir runs/er_buf$B
done

In [ ]:
%%script false --no-raise-error
# REPRODUCE — frozen-feature brittleness control (LSTM vs MLP vs invariant reference). ~40 min
.venv/bin/python scripts/brittleness.py

---
# 5. Results (regenerated from `runs/`)

These cells read the saved JSON. They run in seconds and are the authoritative tables.

## 5.0 The table has a built-in control: diagonal accuracy is flat

Before any comparison: **diagonal accuracy — each task measured immediately after
training it — is flat at 0.970–0.978 across every configuration below** (0.75pp total
spread, ±0.0005–0.0014 within each row), while final retention spans 0.296–0.974
(67.8pp). **Retention variance is 91× the diagonal variance.**

Every method learns every task to the same standard. The entire spread of this table is
a *retention* phenomenon measured with plasticity held constant — which forecloses the
most common ambiguity in continual-learning results, *"did the winner just trade
learning capacity for stability?"*, **by inspection**. Reporting the variance alongside
the mean is what makes that airtight: DIAG is flat *and tight*, so "flat" cannot be
dismissed as "similar on average."

Consequently **RET is the honest headline column**; AVG is retained because readers
expect it, but AVG mixes plasticity and retention while RET isolates what differs.

In [ ]:
def agg(paths):
    v = []
    for p in paths:
        p = ROOT / p
        if not p.exists(): continue
        d = json.load(open(p)); M = np.array(d["accuracy_matrix"])
        v.append([d["average_accuracy"], M[-1, :-1].mean(), np.diag(M).mean(), d["forgetting"]])
    return np.array(v)

def row(label, paths, note=""):
    a = agg(paths)
    if len(a) == 0:
        print(f"{label:<34}  (missing)"); return
    sd = f"+/-{a[:,0].std():.4f}" if len(a) > 1 else "       "
    print(f"{label:<34}{len(a):>3}{a[:,0].mean():>9.4f} {sd}{a[:,1].mean():>9.4f}{a[:,2].mean():>8.4f}{a[:,3].mean():>9.4f}  {note}")

# The built-in control, quantified.
rows_ = [
    ("vanilla LSTM",       ["runs/lstm_results.json"] + [f"runs/lstm_seed{s}/lstm_results.json" for s in (1337,2024)]),
    ("PLCM no adapters",   [f"runs/mafc_attrib/noadapt_seed{s}/mafc_results.json" for s in (42,1337,2024)]),
    ("PLCM+EWC l=200",     ["runs/sweep_200/plcm_ewc_results.json"] + [f"runs/ewc_seed{s}/plcm_ewc_results.json" for s in (1337,2024)]),
    ("ER 100/task",        [f"runs/er_seed{s}/mafc_results.json" for s in (42,1337,2024)]),
    ("adapters unfrozen",  ["runs/mafc_phase1/lambda0/mafc_results.json"] + [f"runs/mafc_phase1/lambda0_seed{s}/mafc_results.json" for s in (1337,2024)]),
    ("adapters frozen",    ["runs/adapter/plcm_adapter_results.json"] + [f"runs/frozen_seed{s}/plcm_adapter_results.json" for s in (1337,2024)]),
]
print(f"{'row':<20}{'DIAG':>19}{'RET':>19}")
print("-" * 58)
Ds, Rs = [], []
for nm, ps in rows_:
    a = agg(ps)
    Ds.append(a[:,2].mean()); Rs.append(a[:,1].mean())
    print(f"{nm:<20}{a[:,2].mean():>10.4f}+/-{a[:,2].std():<7.4f}{a[:,1].mean():>10.4f}+/-{a[:,1].std():<7.4f}")
print("-" * 58)
print(f"  DIAG spread across rows: {(max(Ds)-min(Ds))*100:.2f}pp")
print(f"  RET  spread across rows: {(max(Rs)-min(Rs))*100:.2f}pp")
print(f"  -> RET spread is {(max(Rs)-min(Rs))/(max(Ds)-min(Ds)):.0f}x the DIAG spread: plasticity held constant.")

In [ ]:
S = (42, 1337, 2024)
print("PERMUTED MNIST — 5 tasks x 10 epochs, task-incremental, LSTM backbone")
print(f"{'configuration':<34}{'n':>3}{'AVG':>9}{'':>8}{'RET':>9}{'DIAG':>8}{'forget':>9}")
print("-" * 96)
row("vanilla LSTM", ["runs/lstm_results.json"] + [f"runs/lstm_seed{s}/lstm_results.json" for s in (1337, 2024)])
row("PLCM, no adapters", [f"runs/mafc_attrib/noadapt_seed{s}/mafc_results.json" for s in S])
row("PLCM+EWC lambda=200", ["runs/sweep_200/plcm_ewc_results.json"] + [f"runs/ewc_seed{s}/plcm_ewc_results.json" for s in (1337, 2024)], "NULL result")
row("Experience Replay 100/task", [f"runs/er_seed{s}/mafc_results.json" for s in S], "stores raw data")
row("adapters, unfrozen", ["runs/mafc_phase1/lambda0/mafc_results.json"] + [f"runs/mafc_phase1/lambda0_seed{s}/mafc_results.json" for s in (1337, 2024)])
row("adapters, FROZEN encoder", ["runs/adapter/plcm_adapter_results.json"] + [f"runs/frozen_seed{s}/plcm_adapter_results.json" for s in (1337, 2024)])

## 5.1 Attribution — the single-variable control

Only `adapters.enabled` differs between these two rows. Encoder unfrozen and shared
readout are held constant on both sides.

In [ ]:
off = agg([f"runs/mafc_attrib/noadapt_seed{s}/mafc_results.json" for s in S])
on  = agg(["runs/mafc_phase1/lambda0/mafc_results.json"] + [f"runs/mafc_phase1/lambda0_seed{s}/mafc_results.json" for s in (1337, 2024)])
lstm_a = agg(["runs/lstm_results.json"] + [f"runs/lstm_seed{s}/lstm_results.json" for s in (1337, 2024)])
print(f"vanilla LSTM        {lstm_a[:,0].mean():.4f}")
print(f"PLCM, no adapters   {off[:,0].mean():.4f}   -> memory machinery {(off[:,0].mean()-lstm_a[:,0].mean())*100:+.2f}pp  (nothing measurable)")
print(f"PLCM + adapters     {on[:,0].mean():.4f}   -> ADAPTERS         {(on[:,0].mean()-off[:,0].mean())*100:+.2f}pp")
print(f"\nper-seed adapter delta: {[round(v*100,1) for v in (on[:,0]-off[:,0])]}")
print(f"retention triples: {off[:,1].mean():.3f} -> {on[:,1].mean():.3f}   DIAG unmoved: {off[:,2].mean():.4f} -> {on[:,2].mean():.4f}")

## 5.2 The decomposition panel — the paper's strongest single result

Two independent generality axes. In both, the **delta shrinks only because the control
gets easier**; the adapter arm does not move.

In [ ]:
def m1(t, pre="mlp"):
    a = agg([f"runs/{pre}_{t}_seed{s}/mafc_results.json" for s in S]); return a[:,0].mean()
conds = [("Permuted / LSTM (reference)", off[:,0].mean(), on[:,0].mean()),
         ("Rotated  / LSTM",             m1("noadapt","rot"), m1("adapt","rot")),
         ("Permuted / MLP",              m1("noadapt"),       m1("adapt"))]
print(f"{'condition':<30}{'OFF':>9}{'ON':>9}{'delta':>10}")
print("-" * 58)
for n_, o_, x_ in conds:
    print(f"{n_:<30}{o_:>9.4f}{x_:>9.4f}{(x_-o_)*100:>9.1f}pp")
ref_off, ref_on = conds[0][1], conds[0][2]
print("\nchange vs the reference condition:")
for n_, o_, x_ in conds[1:]:
    print(f"  {n_:<26} OFF {(o_-ref_off)*100:+6.1f}pp    ON {(x_-ref_on)*100:+6.1f}pp")
arms = [c[2] for c in conds]
print(f"\nADAPTER ARM across 2 architectures x 2 shift types: {[round(a,4) for a in arms]}")
print(f"spread = {(max(arms)-min(arms))*100:.1f}pp")

## 5.3 Storage–accuracy frontier

Everything on one basis: bytes **retained after training**. The frontier **crosses twice**,
so the honest claim is regime-dependent.

In [ ]:
d_, T_ = 784, 5
mb_adapt = lambda r: (T_*d_*d_ if r == 0 else T_*2*d_*r) * 4 / 1e6
mb_er    = lambda b: (T_*b*d_) * 4 / 1e6
pts = [
    ("adapters r32", mb_adapt(32), agg(["runs/lowrank_r32/mafc_results.json"] + [f"runs/lowrank_r32_seed{s}/mafc_results.json" for s in (1337,2024)]), "parameters"),
    ("ER 100/task",  mb_er(100),   agg([f"runs/er_seed{s}/mafc_results.json" for s in S]),      "raw images"),
    ("adapters r64", mb_adapt(64), agg(["runs/lowrank_r64/mafc_results.json"] + [f"runs/lowrank_r64_seed{s}/mafc_results.json" for s in (1337,2024)]), "parameters"),
    ("ER 200/task",  mb_er(200),   agg(["runs/er_buf200/mafc_results.json"]),                   "raw images"),
    ("ER 500/task",  mb_er(500),   agg(["runs/er_buf500/mafc_results.json"]),                   "raw images"),
    ("adapters full",mb_adapt(0),  agg(["runs/fullrank_ref/mafc_results.json"]),                "parameters"),
]
pts = [p for p in pts if len(p[2])]
pts.sort(key=lambda p: p[1])
print(f"{'method':<18}{'MB':>7}{'AVG':>9}{'n':>4}   retains")
print("-" * 56)
for n_, mb, a, what in pts:
    print(f"{n_:<18}{mb:>7.2f}{a[:,0].mean():>9.4f}{len(a):>4}   {what}")
print("\nTwo crossings: replay wins at the smallest budget; adapters win from ~2 MB up.")

## 5.4 Frozen-feature brittleness

Encoder trained on task 0, frozen, then a **fresh linear head per task**. No adapters,
no memory, no replay. The raw-pixel arm is a **permutation-invariant reference**, not a
floor: a fresh per-task linear map absorbs any permutation by construction.

In [ ]:
p = ROOT / "runs/brittleness/summary.json"
if p.exists():
    b = json.load(open(p))
    print(f"LSTM frozen transfer   {b['lstm'][0]:.4f} ± {b['lstm'][1]:.4f}")
    print(f"MLP  frozen transfer   {b['mlp'][0]:.4f} ± {b['mlp'][1]:.4f}")
    print(f"invariant reference    {b['raw_invariant_reference']:.4f}")
    print(f"\ngap (MLP - LSTM) = {b['gap_pp']:+.2f}pp  ->  {b['branch']}")
    print(f"joint finding fired: {b['joint_finding']}")
    print("\nBoth encoders sit 16-19pp BELOW a reference that is invariant by construction.")
    print("The failure is representational COMMITMENT, not recurrence:")
    print("each encoder beats the reference on its own task (+6pp) and loses off it (-16..-19pp).")
else:
    print("run scripts/brittleness.py first")

---
# 6. Process appendix — thirteen errors caught before print

The methodology is part of the contribution. Every experiment was **pre-registered**:
hypotheses, thresholds, and decision branches fixed in `docs/*_prereg.md` *before* the
code existed, so results could not be quietly reinterpreted.

**Three distinct mechanisms caught errors, and none was sufficient alone:**

| mechanism | catches | examples |
|---|---|---|
| **Contracts** (pre-registration review) | design errors, *before* compute | H1 unsatisfiable by construction (perfect transport would score at the mush baseline); reversed KL direction; H2 vacuous (identical arms); a 15pp threshold that required >100% accuracy |
| **Baselines** (running the comparison) | claim errors, *after* compute | variance-collapse retracted as mechanism evidence (ER is tighter *without* isolation); adapters cost 10× more storage than the buffer they beat |
| **Execution** (running the code) | integration errors | `LowRankAdapter` has no `.weight`, crashing the per-epoch diagnostic — now a regression test |

## The n=1 rule, and why it exists

**Six configurations were first measured at a single seed. All six moved on seeding.**

| configuration | n=1 | n=3 | shift |
|---|---|---|---|
| adapters, unfrozen | 0.9470 | 0.9331 | −1.4pp |
| vanilla LSTM | 0.4756 | 0.4399 | −3.6pp |
| **EWC λ=200** | **0.4950** | **0.4330** | **−6.2pp** |
| low-rank r32 | 0.7047 | 0.6877 | −1.7pp |
| low-rank r64 | 0.8388 | 0.8719 | **+3.3pp** |

Five optimistic, one pessimistic — **the correction is of unknown sign**, not a haircut
applied to good news.

### The worst instance was invisible until the baseline was seeded

The EWC λ=200 "optimum" was selected at **n=1** from {50, 100, 150, 200, 300}. Seeding
revealed **±0.044** spread on that configuration — *larger than the entire spread across
λ values*. The whole sweep was selecting on noise, and the apparent peak was an artifact.
This is why any hyperparameter that feeds a reported comparison must be chosen at n≥3.

## Claims that were struck

Being able to delete a claim is the point of the process.

1. **"Frozen recurrent features fail because sequential processing bakes in arrangement."**
   Gap was +3.6pp against a ≥15pp threshold → struck. Replaced by the stronger, general
   finding: *any* committed representation fails, feedforward nearly as much as recurrent.
2. **"The adapter represents $P_k^{-1}$."** Falsified twice (Rotated; low-rank) — the design
   outran its own theory. *We built it for the invertible case; it turned out not to need
   invertibility.*
3. **"EWC bought +1.9pp."** An n=1 artifact; the true effect is zero.
4. **"Memory is load-bearing."** Zero across four independent tests.

---
# 7. Honest limitations

- **Task-incremental only.** Per-task adapters require task identity at evaluation. The
  task-free routing attempt reached **20%** accuracy (chance) — the memory bank could not
  supply the routing signal, though a linear probe recovers task identity from cell states
  at 0.995, so the information is present and the *index* is the bottleneck.
- **Parameter growth is `O(T·d²)`** — 614K parameters per task, quadratic in input
  dimension. It does not scale to images: a CIFAR adapter (d=3072) would be 9.4M
  parameters each. Low-rank was tested and is capacity-bound (§3.3).
- **Storage, not efficiency.** Adapters retain no raw data, but full-rank adapters cost
  12.29 MB against a 1.57 MB replay buffer. The privacy claim holds; the efficiency claim
  does not.
- **Input-space shifts only.** Permutation and rotation are both input-space
  transformations. Nothing here tests semantic shift (e.g. Split-CIFAR), where the
  mechanism has no reason to apply.
- **Isolation, not mitigation.** This does not make shared weights resist interference.
  Each task's gradients route through its own input map, so the encoder is *shielded* by
  construction. That is a different achievement from solving interference, and the paper
  says so.